# SKENARIO 2 — Model CoT Terbaik di Dataset Berbeda

Model **fine-tune CoT terbaik** (dari S1, base + adapter LoRA) diuji di beberapa **test set** (numglue, un) -> lihat generalisasi lintas-dataset.

**Sampling** (N=5 kandidat/soal, temp 0.7), ekstrak `\boxed{}`, cocokkan ke gold (answer_check). Butuh **GPU**.
Metrik: **pass@1, pass@2, pass@3, maj@3, maj@5**. Output: tabel per metrik × test set.

## 0. Setup + install peft

In [ ]:
!pip install -q -U peft transformers accelerate

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = '/kaggle/working/FP_NLP'
URL  = 'https://github.com/henray404/FP_NLP.git'
if os.path.exists(REPO):
    subprocess.run(['git','-C',REPO,'fetch','-q','origin','main'], check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard','origin/main'], check=True)
else:
    subprocess.run(['git','clone','-q',URL,REPO], check=True)
sys.path.insert(0, REPO); os.chdir(REPO)
print('repo ready:', REPO)

## 1. Config — base, adapter CoT terbaik, test set

Add Input: dataset berisi adapter hasil training (`adapter_cot_1.5b/`). Isi `ADAPTER_COT` ke path-nya.
Test set dicari otomatis (numglue/un); yang tak ada dilewati.

In [ ]:
import glob
BASE = 'Qwen/Qwen2.5-1.5B'

def find_adapter(keyword):
    hits = glob.glob(f'/kaggle/input/**/*{keyword}*/adapter_config.json', recursive=True)
    return str(Path(hits[0]).parent) if hits else None

ADAPTER_COT = find_adapter('adapter_cot')   # atau isi manual path adapter CoT terbaik
assert ADAPTER_COT, 'adapter CoT tak ketemu -- Add dataset adapter, atau isi ADAPTER_COT manual'
print('adapter CoT:', ADAPTER_COT)

# Test set: cari numglue/un; easy ikut kalau ada. Yang tak ada -> dilewati otomatis.
def find_set(keyword):
    for pat in [f'/kaggle/input/**/*{keyword}*test*.jsonl', f'data/sft/test/*{keyword}*.jsonl']:
        hits = glob.glob(pat, recursive=True)
        if hits: return hits[0]
    return f'data/sft/test/{keyword}_test.jsonl'

SET_PATHS = {'numglue': find_set('numglue'), 'un': find_set('un'), 'easy': find_set('easy')}
print('test sets:', SET_PATHS)

## 2. Eval + tabel

In [ ]:
from src.eval.scenario_eval import load_sets
from src.eval.sample_eval import eval_specs_sampling
from src.eval.sampling_metrics import render_tables
import json

sets = load_sets(SET_PATHS)
specs = {'CoT-terbaik': {'model': BASE, 'adapter': ADAPTER_COT}}
METRICS = ['pass@1', 'pass@2', 'pass@3', 'maj@3', 'maj@5']

# 5 sampel/soal (temp 0.7) -> cukup utk pass@3 & maj@5. Batch per-SOAL; tiap soal mekar x5.
results = eval_specs_sampling(specs, sets, n_samples=5, ks_pass=(1, 2, 3), ks_maj=(3, 5),
                             temperature=0.7, top_p=0.95, max_new_tokens=2048, batch_size=4)

print('\n=== SKENARIO 2: generalisasi lintas-dataset (sampling) ===')
print(render_tables(results, METRICS))
Path('data/eval').mkdir(parents=True, exist_ok=True)
Path('data/eval/s2_crossdata.json').write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')
print('\nsummary -> data/eval/s2_crossdata.json')